In [8]:
import pandas as pd
import numpy as np
%config IPCompliter.greedy=True

In [9]:
train = pd.read_csv("../data/raw/train.csv")
test  = pd.read_csv("../data/raw/test.csv")

# target → 0/1
train["Churn"] = (train["Churn"] == "Yes").astype(int)

In [10]:
def engineer_features(df):
    df = df.copy()

    add_services = [
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies"
    ]

    # --- очистка ---
    for col in add_services:
        df[col] = df[col].replace("No internet service", "No")

    df["has_internet"] = (df["InternetService"] != "No").astype(int)
    df["has_phone"] = (df["PhoneService"] == "Yes").astype(int)

    # --- числовые ---
    df["avg_charge"] = df["TotalCharges"] / df["tenure"].replace(0, np.nan)
    df["avg_spend"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["charges_diff"] = df["TotalCharges"] - df["MonthlyCharges"] * df["tenure"]

    df["monthly_to_total_ratio"] = df["MonthlyCharges"] / (df["TotalCharges"] + 1)
    df["tenure_to_monthly_ratio"] = df["tenure"] / (df["MonthlyCharges"] + 1)
    df["total_to_monthly_ratio"] = df["TotalCharges"] / (df["MonthlyCharges"] + 1)

    df["monthly_minus_avg_spend"] = df["MonthlyCharges"] - df["avg_spend"]

    df["log_monthly"] = np.log1p(df["MonthlyCharges"])
    df["log_total"] = np.log1p(df["TotalCharges"])
    df["log_tenure"] = np.log1p(df["tenure"])

    # --- счетчики ---
    df["num_add_services"] = (df[add_services] == "Yes").sum(axis=1)

    df["has_streaming"] = (
        (df["StreamingTV"] == "Yes") | (df["StreamingMovies"] == "Yes")
    ).astype(int)

    df["streaming_bundle"] = (
        (df["StreamingTV"] == "Yes").astype(int) +
        (df["StreamingMovies"] == "Yes").astype(int)
    )

    df["security_bundle"] = (
        (df["OnlineSecurity"] == "Yes").astype(int) +
        (df["OnlineBackup"] == "Yes").astype(int) +
        (df["DeviceProtection"] == "Yes").astype(int) +
        (df["TechSupport"] == "Yes").astype(int)
    )

    # --- бизнес флаги ---
    df["monthly_contract"] = (df["Contract"] == "Month-to-month").astype(int)
    df["long_contract"] = df["Contract"].isin(["One year", "Two year"]).astype(int)

    df["is_electronic_check"] = (df["PaymentMethod"] == "Electronic check").astype(int)

    df["is_auto_payment"] = df["PaymentMethod"].isin([
        "Bank transfer (automatic)",
        "Credit card (automatic)"
    ]).astype(int)

    df["paperless_and_echeck"] = (
        (df["PaperlessBilling"] == "Yes") &
        (df["PaymentMethod"] == "Electronic check")
    ).astype(int)

    df["is_new_customer"] = (df["tenure"] <= 6).astype(int)
    df["is_mid_customer"] = df["tenure"].between(7, 24).astype(int)
    df["is_loyal_customer"] = (df["tenure"] > 24).astype(int)

    # --- биннинг ---
    df["monthly_bin"] = pd.cut(
        df["MonthlyCharges"],
        bins=[-np.inf, 35, 70, 100, np.inf],
        labels=["low", "mid", "high", "very_high"]
    ).astype(str)

    df["tenure_bin"] = pd.cut(
        df["tenure"],
        bins=[-np.inf, 6, 12, 24, 48, np.inf],
        labels=["very_new", "new", "mid", "long", "very_long"]
    ).astype(str)

    df["services_bin"] = pd.cut(
        df["num_add_services"],
        bins=[-np.inf, 0, 2, 4, 6, np.inf],
        labels=["none", "few", "medium", "many", "very_many"]
    ).astype(str)

    # --- пересечения ---
    df["contract_payment"] = df["Contract"].astype(str) + "_" + df["PaymentMethod"].astype(str)
    df["internet_contract"] = df["InternetService"].astype(str) + "_" + df["Contract"].astype(str)
    df["payment_internet"] = df["PaymentMethod"].astype(str) + "_" + df["InternetService"].astype(str)

    df["tenure_contract"] = df["tenure_bin"].astype(str) + "_" + df["Contract"].astype(str)
    df["monthly_contract_cross"] = df["monthly_bin"].astype(str) + "_" + df["Contract"].astype(str)

    df["internet_payment_contract"] = (
        df["InternetService"].astype(str) + "_" +
        df["PaymentMethod"].astype(str) + "_" +
        df["Contract"].astype(str)
    )

    return df

In [11]:
train = engineer_features(train)
test = engineer_features(test)

In [14]:
cat_cols = train.select_dtypes(include=["object", "str"]).columns.tolist()

num_cols = train.select_dtypes(include=["int64", "float64"]).columns.tolist()
num_cols.remove("Churn")
num_cols.remove("id")

In [15]:
train.shape

(594194, 54)

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from lightgbm import LGBMClassifier

# OHE — более “чистый” вариант, чем замена Yes/No на 1/0
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ]
)

model = LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
    device_type="gpu",
    max_bin=63
)
)

pipe = Pipeline([
    ("prep", preprocessor),
    ("model", model)
])

In [7]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

X = train.drop(columns=["Churn"])
y = train["Churn"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof = np.zeros(len(train))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    pipe.fit(X_tr, y_tr)

    oof[va_idx] = pipe.predict_proba(X_va)[:, 1]

    print(f"Fold {fold} AUC:", roc_auc_score(y_va, oof[va_idx]))

print("OOF AUC:", roc_auc_score(y, oof))

[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.048448 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1486
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567


D:\ml_projects\venvs\churn_s6e3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 1 AUC: 0.9157829064208272
[LightGBM] [Info] Number of positive: 107054, number of negative: 368301
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.078310 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1486
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225209 -> initscore=-1.235567
[LightGBM] [Info] Start training from score -1.235567


D:\ml_projects\venvs\churn_s6e3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 2 AUC: 0.9167048826613013
[LightGBM] [Info] Number of positive: 107053, number of negative: 368302
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.078111 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1486
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225206 -> initscore=-1.235579
[LightGBM] [Info] Start training from score -1.235579


...

Fold 3 AUC: 0.9161380837759177
[LightGBM] [Info] Number of positive: 107053, number of negative: 368302
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.062048 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1486
[LightGBM] [Info] Number of data points in the train set: 475355, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225206 -> initscore=-1.235579
[LightGBM] [Info] Start training from score -1.235579


D:\ml_projects\venvs\churn_s6e3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 4 AUC: 0.9172773816920399
[LightGBM] [Info] Number of positive: 107054, number of negative: 368302
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.144704 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1486
[LightGBM] [Info] Number of data points in the train set: 475356, number of used features: 72
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.225208 -> initscore=-1.235570
[LightGBM] [Info] Start training from score -1.235570


D:\ml_projects\venvs\churn_s6e3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 5 AUC: 0.9145398935187254
OOF AUC: 0.9160816663602087
